In [ ]:
# bibliotecas que serão usadas
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import DecimalType


In [ ]:
# código para o python integrar com o pyspark e acha-lo no disco

import os
import sys
import duckdb
import pyarrow

# Alerta o Spark sobre onde o "winutils.exe" está guardado
os.environ["HADOOP_HOME"] = r"C:\hadoop-3.4.3"
sys.path.append(r"C:\hadoop-3.4.3\bin")

In [6]:
# Sessão do spark
spark = SparkSession.builder.getOrCreate()

In [ ]:
# base que será usada
df = spark.read.option("header", "True").option("sep", ";").csv("C:\\Users\\jv001\\Downloads\\Python\\gastos_consolidado.csv", header=True, inferSchema=True)

In [ ]:
# mostrando a estrutura do df
print(df.show(n=5, truncate=False))

+-----------------------+---------------------+------------------------------------------+------------------------+--------------------------------------------------------------------------+----------------------+---------------------------------------------+-------------+---------------------------------------------+---------------------------+---------------------------------------------+-------------+--------------------+---------------+-------------------------------------+----------------------------+----------------------------------------------------------------------------------------------+-----------+-------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+-------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+---------------------+----+---------+---------------

In [9]:
# selecionando somente as colunas que serão utilizadas

df_reduzido = df.select(
    col("Ano e mês do lançamento").alias("ano_mes_do_lançamento"),
    col("Código Órgão Superior").alias("codigo_orgao_superior"),
    col("Nome Órgão Superior").alias("nome_orgao_superior"),
    col("Código Função").alias("codigo_funcao"),
    col("Nome Função").alias("nome_funcao"),
    col("Código Elemento de Despesa").alias("codigo_elemento_despesa"),
    col("Nome Elemento de Despesa").alias("nome_elemento_despesa"),
    col("Valor Empenhado (R$)").alias("valor_empenhado"),
    col("Valor Liquidado (R$)").alias("valor_liquidado"),
    col("Valor Pago (R$)").alias("valor_pago")
    )
df_reduzido.show(1)

+---------------------+---------------------+--------------------+-------------+-----------+-----------------------+---------------------+---------------+---------------+------------+
|ano_mes_do_lançamento|codigo_orgao_superior| nome_orgao_superior|codigo_funcao|nome_funcao|codigo_elemento_despesa|nome_elemento_despesa|valor_empenhado|valor_liquidado|  valor_pago|
+---------------------+---------------------+--------------------+-------------+-----------+-----------------------+---------------------+---------------+---------------+------------+
|              2015/01|                26000|Ministério da Edu...|           12|   Educação|                     41|        Contribuições|  7898910232,69|   552807273,04|552807273,04|
+---------------------+---------------------+--------------------+-------------+-----------+-----------------------+---------------------+---------------+---------------+------------+
only showing top 1 row


In [10]:
# tratar colunas de valores e data
df_tratado = df_reduzido \
    .withColumn("valor_empenhado", regexp_replace(col("valor_empenhado"), ",", ".")) \
    .withColumn("valor_liquidado", regexp_replace(col("valor_liquidado"), ",", ".")) \
    .withColumn("valor_pago", regexp_replace(col("valor_pago"), ",", ".")) \
    .withColumn("ano_mes_do_lançamento", to_date(concat(col("ano_mes_do_lançamento"), lit(r"/01")), r"yyyy/MM/dd")) 
    
df_tratado = df_tratado \
    .withColumn("valor_empenhado", col("valor_empenhado").cast(DecimalType(15, 2))) \
    .withColumn("valor_liquidado", col("valor_liquidado").cast(DecimalType(15, 2 ))) \
    .withColumn("valor_pago", col("valor_pago").cast(DecimalType(15, 2)))
    

df_tratado.show(5)

+---------------------+---------------------+--------------------+-------------+--------------------+-----------------------+---------------------+---------------+---------------+------------+
|ano_mes_do_lançamento|codigo_orgao_superior| nome_orgao_superior|codigo_funcao|         nome_funcao|codigo_elemento_despesa|nome_elemento_despesa|valor_empenhado|valor_liquidado|  valor_pago|
+---------------------+---------------------+--------------------+-------------+--------------------+-----------------------+---------------------+---------------+---------------+------------+
|           2015-01-01|                26000|Ministério da Edu...|           12|            Educação|                     41|        Contribuições|  7898910232.69|   552807273.04|552807273.04|
|           2015-01-01|                41000|Ministério das Co...|           24|        Comunicações|                     39| Outros Serviços d...|      628945.96|       31351.72|    31351.72|
|           2015-01-01|            

In [ ]:
# verificando nulos
qtd_nulo = [
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_tratado.columns
    ]

df_tratado.select(qtd_nulo).show()
df_tratado.count()

+---------------------+---------------------+-------------------+-------------+-----------+-----------------------+---------------------+---------------+---------------+----------+
|ano_mes_do_lançamento|codigo_orgao_superior|nome_orgao_superior|codigo_funcao|nome_funcao|codigo_elemento_despesa|nome_elemento_despesa|valor_empenhado|valor_liquidado|valor_pago|
+---------------------+---------------------+-------------------+-------------+-----------+-----------------------+---------------------+---------------+---------------+----------+
|                    0|                    0|                  0|            0|          0|                      0|                    0|              0|              0|         0|
+---------------------+---------------------+-------------------+-------------+-----------+-----------------------+---------------------+---------------+---------------+----------+



In [ ]:
# algumas linhas tem as colunas de valores todas em 0
# filtrar o df sem essas linhas
df_filtro = df_tratado.filter(
    (col("valor_empenhado") > 0) |
    (col("valor_liquidado") > 0 ) |
    (col("valor_pago") > 0)
)

df_filtro.show(10)
df_filtro.count()

+---------------------+---------------------+--------------------+-------------+--------------------+-----------------------+---------------------+---------------+---------------+------------+
|ano_mes_do_lançamento|codigo_orgao_superior| nome_orgao_superior|codigo_funcao|         nome_funcao|codigo_elemento_despesa|nome_elemento_despesa|valor_empenhado|valor_liquidado|  valor_pago|
+---------------------+---------------------+--------------------+-------------+--------------------+-----------------------+---------------------+---------------+---------------+------------+
|           2015-01-01|                26000|Ministério da Edu...|           12|            Educação|                     41|        Contribuições|  7898910232.69|   552807273.04|552807273.04|
|           2015-01-01|                41000|Ministério das Co...|           24|        Comunicações|                     39| Outros Serviços d...|      628945.96|       31351.72|    31351.72|
|           2015-01-01|            

In [ ]:
# retirando os espaços das colunas de texto
df_final = df_filtro \
    .withColumn("nome_orgao_superior", trim(col("nome_orgao_superior"))) \
    .withColumn("nome_funcao", trim(col("nome_funcao"))) \
    .withColumn("nome_elemento_despesa", trim(col("nome_elemento_despesa")))

In [ ]:
# salvar df tratado

pasta = "C:\\Users\\jv001\\Downloads\\Python\\curso_ebac\\projeto_semantrix\\despesas_tratadas.parquet"

arrow_table = df_final.toArrow()
con = duckdb.connect()
con.register("view_spark", arrow_table)

con.execute(f"COPY view_spark TO '{pasta}' (FORMAT PARQUET);")

print("arquivo salvo")



arquivo salvo


In [15]:
spark.stop()